# Experiment 1: Llama-3.2-3B Generation Quality

**Goal:** Fill the last model gap in the generation quality evaluation.
- Llama-3.2-3B is the only model missing from `generation_quality_fp16_merged`
- Tests 7 compressors × 50 WikiText prompts = 350 results
- **Runtime:** ~2-3 hours on T4

## Setup
1. Create zip locally: `cd KVShuttle && zip -r kvshuttle.zip . -x '.git/*' -x '*.pyc' -x '__pycache__/*'`
2. Set runtime to GPU (T4 is sufficient — Llama-3.2-3B is only 3B params)
3. Run all cells — upload zip when prompted
4. **Important:** Paste your HF token (Llama requires license acceptance)

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Go to Runtime > Change runtime type > GPU")

In [ ]:
# Install dependencies + upload KVShuttle
!pip install -q transformers accelerate datasets pyyaml tqdm

import os
if not os.path.exists("kvshuttle"):
    from google.colab import files
    print("Upload kvshuttle.zip")
    uploaded = files.upload()
    !unzip -qo kvshuttle.zip -d KVShuttle
    %cd KVShuttle
    !pip install -q -e .
else:
    print("KVShuttle already installed")

In [ ]:
# HuggingFace login (required for Llama gated models)
# Get token from https://huggingface.co/settings/tokens
# Also accept license at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
from huggingface_hub import login
login(token="PASTE_YOUR_HF_TOKEN_HERE")

In [ ]:
# Verify installation
from kvshuttle.models.loader_torch import TORCH_MODEL_REGISTRY
from kvshuttle.compression.registry import list_compressors
print(f"Models: {list(TORCH_MODEL_REGISTRY.keys())}")
print(f"Compressors: {list_compressors()}")
assert 'llama-3.2-3b' in TORCH_MODEL_REGISTRY, "llama-3.2-3b not in registry!"

In [ ]:
# Write config inline and run experiment
import yaml
from pathlib import Path

config = {
    "experiment": {
        "name": "generation_quality_fp16",
        "description": "Llama-3.2-3B FP16 generation quality on T4 GPU"
    },
    "backend": "torch",
    "models": ["llama-3.2-3b"],
    "compressors": [
        "identity", "uniform_int8", "uniform_int4",
        "kivi_2bit", "cachegen", "palu_lr", "cascade_prune50_int4"
    ],
    "bandwidths_gbps": [10],
    "prompts": {
        "source": "wikitext",
        "count": 50,
        "min_tokens": 128,
        "max_tokens": 512
    },
    "evaluation": {
        "attention_error": True,
        "perplexity": True,
        "token_agreement": True
    },
    "output": {
        "dir": "experiments/results/generation_quality_fp16_llama32",
        "save_per_layer": False
    }
}

config_path = Path("experiments/configs/generation_quality_torch_llama32.yaml")
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, "w") as f:
    yaml.dump(config, f)
print(f"Config: {config_path}")
print(f"Models: {config['models']}")
print(f"Compressors: {config['compressors']}")
print(f"Expected results: {len(config['models'])} × {len(config['compressors'])} × {config['prompts']['count']} = {len(config['models']) * len(config['compressors']) * config['prompts']['count']}")

!python -m experiments.scripts.run_experiment {config_path}

In [ ]:
# Inspect results
import json
import pandas as pd
from pathlib import Path

results_path = Path("experiments/results/generation_quality_fp16_llama32/results.json")

if results_path.exists():
    with open(results_path) as f:
        data = json.load(f)
    print(f"Results: {len(data['results'])} entries")

    df = pd.DataFrame(data['results'])
    agg = df.groupby(['model', 'compressor']).agg({
        'mean_key_cosine_sim': 'mean',
        'token_agreement': 'mean',
        'perplexity_delta': 'mean',
        'compression_ratio': 'mean'
    }).round(4)
    display(agg)
else:
    print("Results not found. Check experiment output above for errors.")

In [ ]:
# Download results
if results_path.exists():
    from google.colab import files
    files.download(str(results_path))
    print(f"Downloaded {results_path}")
    print("\nNext steps:")
    print("1. Copy results.json to experiments/results/generation_quality_fp16_llama32/")
    print("2. Run: python experiments/scripts/merge_fp16_results_v2.py")
    print("3. Run: python experiments/scripts/compute_confidence_intervals.py")